In [14]:
import torch
import numpy as np

In [15]:
from dcl_definitions_cuda import Classifier, LostSalesEnv
# --- 1. 장치 설정 및 체크포인트 파일 불러오기 ---

# 사용할 장치를 명시적으로 정의
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"모델을 불러와서 사용할 장치: {device}")

# 저장했던 체크포인트 파일을 불러옵니다.
# 파일 이름은 실제 저장된 파일 이름으로 변경해야 합니다.
checkpoint = torch.load('dcl_lost_sales_checkpoint_YYYYMMDD_HHMMSS.pth', map_location=device)


# --- 2. 저장된 하이퍼파라미터 출력하기 ---

dcl_hyperparams = checkpoint['dcl_hyperparameters']
env_params = checkpoint['env_parameters']

print("\n" + "="*50)
print("             모델 훈련에 사용된 파라미터 정보")
print("="*50)
print("\n[DCL 하이퍼파라미터]")
for key, value in dcl_hyperparams.items():
    print(f"  - {key}: {value}")

print("\n[환경 파라미터]")
for key, value in env_params.items():
    print(f"  - {key}: {value}")
print("="*50)


# --- 3. 모델 가중치 불러오기 ---

# 저장된 파라미터를 기반으로 모델의 껍데기를 정확하게 다시 만듭니다.
env = LostSalesEnv(lead_time=env_params['lead_time'], 
                   holding_cost=env_params['holding_cost'], 
                   penalty_cost=env_params['penalty_cost'], 
                   mean_demand=env_params['mean_demand'], 
                   max_action=env_params['max_action'])

loaded_policy_lost_sales = Classifier(env.state_size, env.num_actions)
loaded_policy_lost_sales.to(device)

# 체크포인트에서 모델 가중치를 불러와 적용합니다.
loaded_policy_lost_sales.load_state_dict(checkpoint['model_state_dict'])
loaded_policy_lost_sales.eval()

print("\n저장된 정책 모델을 성공적으로 불러왔습니다!")


# --- 4. 불러온 모델 사용하기 ---

example_state = np.array([5, 10, 8, 5])
# 안전장치를 위해 상태 벡터 차원을 검사합니다.
if len(example_state) != env.state_size:
    print(f"\n[오류] 입력된 상태 벡터의 차원({len(example_state)})이 모델이 기대하는 차원({env.state_size})과 다릅니다.")
else:
    action = loaded_policy_lost_sales.get_action(example_state)
    print(f"\n불러온 모델의 결정 행동: {action}")

모델을 불러와서 사용할 장치: cuda
저장된 정책 모델을 성공적으로 불러왔습니다!


C:\Users\GMS\AppData\Local\Temp\ipykernel_10216\3720282368.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_policy_lost_sales.load_state_dict(torch.load('dcl_lost

In [17]:
# 이 셀을 실행하기 전에 DCL 학습이 완료되어 Lost Sales에 대한 'best_policy' 변수가 생성되어 있어야 합니다.


print("학습된 Lost Sales 정책 모델에게 여러가지 재고 상황에 대해 질문합니다.")
print("-" * 50)

# --- 시나리오 1: 재고가 하나도 없는 상황 ---
state_1 = np.array([0, 0, 0])
action_1 = loaded_policy_lost_sales.get_action(state_1)
print(f"상황 1: 재고가 전혀 없을 때 {state_1}")
print(f"▶ 추천 주문량: {action_1} 개")
print("-" * 50)


# --- 시나리오 2: 재고는 적지만, 곧 많이 들어올 예정인 상황 ---
state_2 = np.array([5, 20, 15]) # 현재고 5, 파이프라인에 45개
action_2 = loaded_policy_lost_sales.get_action(state_2)
print(f"상황 2: 재고는 적지만 파이프라인이 꽉 찼을 때 {state_2}")
print(f"▶ 추천 주문량: {action_2} 개 (아마 0에 가까울 것입니다)")
print("-" * 50)


# --- 시나리오 3: 재고가 매우 많은 상황 ---
state_3 = np.array([50, 0, 0])
action_3 = loaded_policy_lost_sales.get_action(state_3)
print(f"상황 3: 재고가 매우 많을 때 {state_3}")
print(f"▶ 추천 주문량: {action_3} 개 (반드시 0이어야 합니다)")
print("-" * 50)

학습된 Lost Sales 정책 모델에게 여러가지 재고 상황에 대해 질문합니다.
--------------------------------------------------
상황 1: 재고가 전혀 없을 때 [0 0 0 0]
▶ 추천 주문량: 5 개
--------------------------------------------------
상황 2: 재고는 적지만 파이프라인이 꽉 찼을 때 [ 5 20 15 10]
▶ 추천 주문량: 5 개 (아마 0에 가까울 것입니다)
--------------------------------------------------
상황 3: 재고가 매우 많을 때 [50  0  0  0]
▶ 추천 주문량: 5 개 (반드시 0이어야 합니다)
--------------------------------------------------


In [19]:
from dcl_definitions_cuda import Classifier, PerishableInventoryEnv

# --- 1. 장치 설정 및 체크포인트 파일 불러오기 ---

# 사용할 장치를 명시적으로 정의
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"모델을 불러와서 사용할 장치: {device}")

# <--- 수정: Perishable Inventory 모델 파일 이름을 사용합니다.
# 파일 이름은 실제 저장된 파일 이름으로 변경해야 합니다.
checkpoint = torch.load('dcl_perishable_checkpoint_YYYYMMDD_HHMMSS.pth', map_location=device)


# --- 2. 저장된 하이퍼파라미터 출력하기 ---

dcl_hyperparams = checkpoint['dcl_hyperparameters']
env_params = checkpoint['env_parameters']

print("\n" + "="*50)
print("             모델 훈련에 사용된 파라미터 정보")
print("="*50)
print("\n[DCL 하이퍼파라미터]")
for key, value in dcl_hyperparams.items():
    print(f"  - {key}: {value}")

print("\n[환경 파라미터]")
for key, value in env_params.items():
    print(f"  - {key}: {value}")
print("="*50)


# --- 3. 모델 가중치 불러오기 ---

# <--- 수정: 저장된 파라미터를 기반으로 'PerishableInventoryEnv' 껍데기를 다시 만듭니다.
env_pi = PerishableInventoryEnv(
    lifetime=env_params['lifetime'],
    lead_time=env_params['lead_time'],
    holding_cost=env_params['holding_cost'],
    penalty_cost=env_params['penalty_cost'],
    waste_cost=env_params['waste_cost'],
    mean_demand=env_params['mean_demand'],
    max_action=env_params['max_action'],
    fifo_ratio=env_params['fifo_ratio']
)

# <--- 수정: 변수 이름을 명확하게 변경합니다.
loaded_policy_perishable = Classifier(env_pi.state_size, env_pi.num_actions)
loaded_policy_perishable.to(device)

# 체크포인트에서 모델 가중치를 불러와 적용합니다.
loaded_policy_perishable.load_state_dict(checkpoint['model_state_dict'])
loaded_policy_perishable.eval()

print("\n저장된 정책 모델을 성공적으로 불러왔습니다!")


# --- 4. 불러온 모델 사용하기 ---

# <--- 수정: Perishable Inventory에 맞는 예시 상태 벡터를 사용합니다.
# (예: lifetime=3, lead_time=2 -> 4차원)
example_state_pi = np.array([2, 5, 8, 10])

# 안전장치를 위해 상태 벡터 차원을 검사합니다.
if len(example_state_pi) != env_pi.state_size:
    print(f"\n[오류] 입력된 상태 벡터의 차원({len(example_state_pi)})이 모델이 기대하는 차원({env_pi.state_size})과 다릅니다.")
else:
    action_pi = loaded_policy_perishable.get_action(example_state_pi)
    print(f"\n불러온 모델의 결정 행동: {action_pi}")

모델을 불러와서 사용할 장치: cuda
저장된 정책 모델을 성공적으로 불러왔습니다!


C:\Users\GMS\AppData\Local\Temp\ipykernel_10216\2940904496.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_policy_perishable.load_state_dict(torch.load('dcl_peri

In [20]:
# --- 2. 불러온 모델 사용하기 ---

# 이제 'loaded_policy'는 훈련된 'best_policy'와 완전히 똑같은 전문가입니다.
example_state = np.array([5, 10, 8, 5])
action = loaded_policy_perishable.get_action(example_state)

print(f"\n새 컴퓨터에서 불러온 모델의 결정 행동: {action}")


새 컴퓨터에서 불러온 모델의 결정 행동: 0


In [21]:
# 이 셀을 실행하기 전에 DCL 학습이 완료되어 Lost Sales에 대한 'best_policy' 변수가 생성되어 있어야 합니다.


print("학습된 erishable 정책 모델에게 여러가지 재고 상황에 대해 질문합니다.")
print("-" * 50)

# --- 시나리오 1: 재고가 하나도 없는 상황 ---
state_1 = np.array([0, 0, 0, 0])
action_1 = loaded_policy_perishable.get_action(state_1)
print(f"상황 1: 재고가 전혀 없을 때 {state_1}")
print(f"▶ 추천 주문량: {action_1} 개")
print("-" * 50)


# --- 시나리오 2: 재고는 적지만, 곧 많이 들어올 예정인 상황 ---
state_2 = np.array([5, 20, 15, 10]) # 현재고 5, 파이프라인에 45개
action_2 = loaded_policy_perishable.get_action(state_2)
print(f"상황 2: 재고는 적지만 파이프라인이 꽉 찼을 때 {state_2}")
print(f"▶ 추천 주문량: {action_2} 개 (아마 0에 가까울 것입니다)")
print("-" * 50)


# --- 시나리오 3: 재고가 매우 많은 상황 ---
state_3 = np.array([50, 0, 0, 0])
action_3 = loaded_policy_perishable.get_action(state_3)
print(f"상황 3: 재고가 매우 많을 때 {state_3}")
print(f"▶ 추천 주문량: {action_3} 개 (반드시 0이어야 합니다)")
print("-" * 50)

학습된 erishable 정책 모델에게 여러가지 재고 상황에 대해 질문합니다.
--------------------------------------------------
상황 1: 재고가 전혀 없을 때 [0 0 0 0]
▶ 추천 주문량: 0 개
--------------------------------------------------
상황 2: 재고는 적지만 파이프라인이 꽉 찼을 때 [ 5 20 15 10]
▶ 추천 주문량: 0 개 (아마 0에 가까울 것입니다)
--------------------------------------------------
상황 3: 재고가 매우 많을 때 [50  0  0  0]
▶ 추천 주문량: 0 개 (반드시 0이어야 합니다)
--------------------------------------------------
